# Lab 5 · Short-Term Memory

The Lab 4 agent answers each question in isolation. Ask it "Tell me about Apple's risk factors" and then "What about their competitors?" and it has no idea who "their" refers to. Every call starts from a blank slate.

Agent memory comes in three connected layers, short-term, long-term, and reasoning traces, that together form a **context graph** stored in Neo4j. This notebook adds the first, a **short-term memory** layer, with [`neo4j-agent-memory`](https://github.com/neo4j-labs/agent-memory). Each turn of the conversation is written to Neo4j as `Conversation` and `Message` nodes. Before every new question we pull the relevant history back out and hand it to the agent, so the continuity lives in the graph rather than in the language model's context window.

**Learning objectives**
- Open a `MemoryClient` against the same Aura instance and Titan embeddings the earlier labs use
- Write each turn to short-term memory with `add_message`
- Inject prior context with `get_context` so the agent resolves references across turns
- See the `Conversation` and `Message` nodes sitting beside the SEC 10-K graph

**Prerequisites:** the seed load from Lab 1 has populated the graph with chunks, embeddings, and the `chunkEmbeddings` vector index, and `CONFIG.txt` holds your Neo4j and AWS credentials.

In [ ]:
%pip install "neo4j-agent-memory[bedrock]==0.5.0" strands-agents "neo4j-graphrag[bedrock]>=1.18.0" -q

## Setup

`neo4j-agent-memory` is asynchronous. Every memory call is a coroutine, so in the notebook we prefix each one with `await`. We reuse the Lab 4 GraphRAG agent unchanged and add the memory layer around it.

In [ ]:
import sys
from pathlib import Path

# Lab 5 lives beside Lab 4. Add both to the import path:
#   this lab's dir   provides lib.data_utils and lib.memory_utils
#   Lab 4's lib dir  provides graphrag_agent, the canonical GraphRAG agent
#                    module, imported here rather than copied so there is one
#                    source of truth.
LAB5_DIR = Path.cwd()
LAB4_LIB = LAB5_DIR.parent / "Lab_4_GraphRAG_Agent" / "lib"
for p in (str(LAB4_LIB), str(LAB5_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Lab 5 dir:  {LAB5_DIR}")
print(f"Lab 4 lib:  {LAB4_LIB}")

In [ ]:
from graphrag_agent import build_graphrag_agent
from lib.memory_utils import build_memory_client

# The known Lab 4 agent: two neo4j-graphrag retrievers wrapped as Strands tools.
graphrag = build_graphrag_agent()
agent = graphrag.agent
print("GraphRAG agent ready.")

# The memory client, over the same Aura instance and Titan embeddings.
memory = build_memory_client()
await memory.connect()
print("Memory client connected.")

## Smoke test

Confirm the memory client can write and read before wiring anything together. We add one message to a throwaway session and ask for its context back.

In [ ]:
SMOKE_SESSION = "lab5-smoke"

await memory.short_term.add_message(
    SMOKE_SESSION, "user", "Apple's largest risk is supply chain concentration."
)
context = await memory.get_context(
    "What is Apple's largest risk?",
    session_id=SMOKE_SESSION,
    include_long_term=False,
    include_reasoning=False,
)
print(context)

## Wrap the agent with memory

The pattern is deliberately explicit, two lines of memory work around each agent call:

1. **Before** the turn, `get_context(question, session_id=...)` pulls back the relevant conversation so far, and we prepend it to the question.
2. **After** the turn, `add_message` writes both the user's question and the agent's answer, so the next turn can see them.

Notice `agent.messages = []` at the top. We clear the agent's in-process history each turn on purpose. Whatever cross-turn continuity you see below then comes entirely from Neo4j-backed memory, not from the model's own context window. That is the pattern that survives a restart and scales across sessions and machines.

In [ ]:
async def ask(question: str, session_id: str) -> str:
    """Answer a question with the GraphRAG agent, backed by short-term memory."""
    # 1. Pull relevant prior conversation and prepend it to the question.
    #    Scope to short-term only so the continuity we see is conversational
    #    recall, not long-term knowledge or reasoning traces (covered later).
    context = await memory.get_context(
        question,
        session_id=session_id,
        include_long_term=False,
        include_reasoning=False,
    )
    prompt = f"{context}\n\nUser question: {question}" if context else question

    # Clear in-process history so continuity comes only from memory (see above).
    agent.messages = []
    response = agent(prompt)
    answer = str(response)

    # 2. Persist both sides of the turn for the next question to build on.
    await memory.short_term.add_message(session_id, "user", question)
    await memory.short_term.add_message(session_id, "assistant", answer)
    return answer

## The headline demo

Three questions in one session. The second never names Apple, and the third refers to the whole conversation. Both only work because the earlier turns were written to memory and injected back in.

In [ ]:
SESSION = "apple-research"

_ = await ask("Tell me about Apple's risk factors.", SESSION)

In [ ]:
_ = await ask("What about their competitors?", SESSION)

In [ ]:
_ = await ask("Summarize what we discussed.", SESSION)

## Inspect the memory in Neo4j

The conversation is now durable in the graph. Here are the `Message` nodes for this session, in order, sitting in the same database as the SEC 10-K `Company` and `Chunk` nodes.

In [ ]:
# memory.query.cypher runs read-only Cypher against the same graph, and
# works on both the bolt and hosted backends.
rows = await memory.query.cypher(
    """
    MATCH (c:Conversation {session_id: $session_id})-[:HAS_MESSAGE]->(m:Message)
    RETURN m.role AS role, m.content AS content, m.timestamp AS timestamp
    ORDER BY m.timestamp
    """,
    {"session_id": SESSION},
)
for r in rows:
    print(f"[{r['role']}] {r['content'][:100]}")

## Cleanup

Close the memory client and the agent's Neo4j driver.

In [ ]:
await memory.close()
graphrag.close()
print("Closed memory client and agent driver.")

## Summary

You wrapped the Lab 4 GraphRAG agent with a short-term memory layer:

| Step | Call | Effect |
|------|------|--------|
| Before a turn | `memory.get_context(q, session_id=...)` | Pull relevant prior conversation |
| After a turn | `memory.short_term.add_message(...)` | Persist the user and assistant messages |

The agent now resolves references like "their" across turns, and the whole conversation is queryable in Neo4j beside the domain graph.

Short-term memory recalls the current conversation. The next notebook adds **long-term memory**: durable entities, facts, and preferences that survive across sessions.

---

**Next:** [`02_long_term_memory.ipynb`](02_long_term_memory.ipynb), durable knowledge across sessions